# 02. Baseline Model: TF-IDF + Logistic Regression

> **Historical / exploratory notebook. The canonical implementation is under `src/`.**

**Môn học:** Trí tuệ Nhân tạo (AI)  
**Đề tài:** Ứng dụng BERT trong phân loại cảm xúc đánh giá khách sạn  
**Phương pháp luận:** Tuân thủ giai đoạn 03 (Models) trong `AI Project Cycle.pptx`:  
> *"We should start from the simple to more complex models because our solutions need to be as compact as possible."*

In [ ]:
import os
import sys
import time
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# Import data module từ src
sys.path.append(os.path.abspath(os.path.join("..", "src")))
from data import load_and_clean_data, split_data

sns.set_theme(style="whitegrid")
print("[*] Modules loaded successfully!")

## 1. Nạp và Phân chia dữ liệu (Stratified Split, seed=42)
Đảm bảo tỷ lệ Train (70%), Val (10%), Test (20%) hoàn toàn cân bằng giữa hai lớp nhãn.

In [ ]:
df = load_and_clean_data()
train_df, val_df, test_df = split_data(df, train_ratio=0.7, val_ratio=0.1, test_ratio=0.2, random_state=42)

## 2. Xây dựng Pipeline TF-IDF + Logistic Regression
- Sử dụng n-gram từ 1 đến 2 từ (unigram + bigram).
- Giới hạn tối đa 10,000 đặc trưng phổ biến nhất.
- `sublinear_tf=True` để làm mượt tần suất từ (thay thế $tf$ bằng $1 + \log(tf)$).

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=10000,
        sublinear_tf=True,
        strip_accents='unicode'
    )),
    ('clf', LogisticRegression(
        C=1.0,
        max_iter=1000,
        random_state=42,
        solver='lbfgs'
    ))
])
print("[*] Pipeline structure:")
print(pipeline)

## 3. Huấn luyện mô hình cơ sở

In [ ]:
start_train = time.time()
pipeline.fit(train_df["text"], train_df["label"])
train_duration = time.time() - start_train
print(f"[+] Huấn luyện xong trong {train_duration:.2f} giây.")

## 4. Đánh giá trên Validation Set

In [ ]:
val_preds = pipeline.predict(val_df["text"])
val_acc = accuracy_score(val_df["label"], val_preds)
val_f1 = f1_score(val_df["label"], val_preds, average='macro')
print(f"[*] Validation Accuracy: {val_acc*100:.2f}%")
print(f"[*] Validation Macro F1: {val_f1:.4f}")

## 5. Đánh giá duy nhất trên Test Set độc lập (Test Evaluation)
Tập Test chỉ được nạp đúng một lần để kiểm chứng hiệu năng thực tế.

In [ ]:
start_eval = time.time()
test_preds = pipeline.predict(test_df["text"])
eval_duration = time.time() - start_eval

acc = accuracy_score(test_df["label"], test_preds)
f1_macro = f1_score(test_df["label"], test_preds, average='macro')
prec_macro = precision_score(test_df["label"], test_preds, average='macro')
rec_macro = recall_score(test_df["label"], test_preds, average='macro')

print("="*50)
print("KẾT QUẢ ĐÁNH GIÁ TRÊN TEST SET:")
print(f"Accuracy:        {acc*100:.2f}%")
print(f"Macro Precision: {prec_macro:.4f}")
print(f"Macro Recall:    {rec_macro:.4f}")
print(f"Macro F1-Score:  {f1_macro:.4f}")
print(f"Inference Speed: {len(test_df)/eval_duration:.1f} samples/sec")
print("="*50)
print("\nClassification Report:")
print(classification_report(test_df["label"], test_preds, target_names=["Negative (0)", "Positive (1)"], digits=4))

## 6. Ma trận nhầm lẫn (Confusion Matrix)

In [ ]:
cm = confusion_matrix(test_df["label"], test_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Negative (0)", "Positive (1)"],
            yticklabels=["Negative (0)", "Positive (1)"],
            annot_kws={"size": 14, "weight": "bold"})
plt.title("Confusion Matrix - TF-IDF + Logistic Regression", fontsize=13, fontweight="bold")
plt.xlabel("Predicted Label", fontsize=11)
plt.ylabel("True Label", fontsize=11)
plt.tight_layout()
plt.show()

## 7. Kết luận về Baseline
- TF-IDF + Logistic Regression là một mô hình cơ sở mạnh, tốc độ cực nhanh.
- Tuy nhiên, hạn chế cố hữu là xem văn bản là "túi từ" (Bag-of-Words), không hiểu được trật tự từ, ngữ cảnh đa chiều và các cấu trúc đảo ngữ phức tạp.
- Đây là cơ sở vững chắc để chứng minh sự cần thiết của mô hình BERT ở các bước tiếp theo.